# 02 – Analysis: set vs graph neural networks for xG

**Question:** does modelling the players around a shot as a **graph** (players connected to each other) predict goals
better than treating them as an unordered **set** (each player seen on its own)?

**Input:** `data/processed/shot_table.csv`, `player_table.csv` and `feature_columns.json` from `01-Data` (the shooter is already
a row of `player_table`). **Output:** tables in `results/`, figures in `figures/` (numbered after the two figures of `01-Data`).

**Reading order**
1. Load and prepare the data
2. How a shot becomes a graph
3. The two models (one shared code path: the set model is the graph model with no message passing)
4. Training, evaluation and **how we deal with noise**
5. Experiments: tuning choices on the validation set (edge rule → region size → aggregation → features → training data)
6. Final test: set vs graph on held-out matches
7. Scaling: what happens with far more data
8. Conclusions (generated from the results above, so the numbers cannot go stale)

**Run time:** roughly 1–1.5 hours on a 2-core laptop CPU in `full` mode (every configuration is trained with three seeds;
this is an estimate, not a measurement). `smoke` mode checks that everything runs in a few minutes.
Set `MODE` below, or the environment variable `XG_MODE`.

In [ ]:
import json, time, copy, os, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.patches import Wedge, Polygon
from scipy.optimize import curve_fit
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score, roc_curve

warnings.filterwarnings("ignore", category=UserWarning)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_THREADS = torch.get_num_threads()
N_CPU = os.cpu_count() or 1
print("torch", torch.__version__, "| device:", DEVICE, "| CPU threads:", N_THREADS, "| CPU cores:", N_CPU)

## 0. Settings
* `MODE = "smoke"`: tiny run to check the code (minutes). `MODE = "full"`: the real results.
* **Every** configuration, in every experiment, is trained with several seeds (`SEEDS`). Earlier tuning used one seed per
  configuration, which cannot tell a real difference from seed noise.
* `EQUIV_MARGIN`: two models are called *equivalent* if the 95% interval of their log-loss difference lies inside ±0.0025
  (about 1% of the log loss). Note that we chose this convention so it is not a set rule.

In [ ]:
MODE = os.environ.get("XG_MODE", "full")   # "smoke" | "full"

SETTINGS = {
    "smoke": dict(seeds=[0, 1], max_epochs=2, max_train=3000, synth=[1, 2], fracs=[0.25, 0.5],
                  profile_batches=5, n_boot=300),
    "full":  dict(seeds=[0, 1, 2], max_epochs=25, max_train=None, synth=[1, 2, 5, 10], fracs=[0.05, 0.1, 0.25, 0.5],
                  profile_batches=40, n_boot=2000),
}
S_ = SETTINGS[MODE]
SEEDS, MAX_EPOCHS, MAX_TRAIN, N_BOOT = S_["seeds"], S_["max_epochs"], S_["max_train"], S_["n_boot"]
PATIENCE = 4               # stop when validation log loss has not improved for 4 epochs
EQUIV_MARGIN = 0.0025      # log-loss difference regarded as practically irrelevant (about 1% of the log loss)

PROC, RESULTS_DIR, FIGURES = Path("data/processed"), Path("results"), Path("figures")
RESULTS_DIR.mkdir(exist_ok=True); FIGURES.mkdir(exist_ok=True)

PITCH_L, PITCH_W = 120.0, 80.0                          # StatsBomb pitch: everyone attacks towards x = 120
GOAL_X, GOAL_Y, POST_L, POST_R = 120.0, 40.0, 36.0, 44.0
print("MODE:", MODE, "| seeds per configuration:", SEEDS)

## 1. Load and prepare the data

**Evaluation sets.** All selection and the headline result use matches from the four **complete 2015/16 league seasons**
(Premier League, La Liga, Serie A, Ligue 1; Ligue 1 2015/16 misses a few matches in the open data but is treated as complete).
In the code and tables these are called *complete seasons*. Single-club seasons are a biased sample (one strong team in every match),
so they are kept only for comparison. Training uses all seasons unless Experiment 5.5 says otherwise.
The flags come from `shot_table` (`subset_1516`, `is_focus_team`); the "focus club" of a single-club season is the team that plays in
every match of that season.

**Nodes.** Each shot becomes a graph with one node per visible player plus the **shooter node** (`is_shooter = 1`, the first row of
each shot in `player_table`; role one-hots are 0 because StatsBomb does not give the shooter's position in the frame).

**Scaling.** Continuous features are standardised with means and standard deviations from the **training split only**.
Binary flags are left as 0/1. The number of players is *not* given as a feature, so that the aggregation experiment (5.3) can show
whether a model is sensitive to how many players there are.

In [ ]:
shot_df = pd.read_csv(PROC / "shot_table.csv")
player_df = pd.read_csv(PROC / "player_table.csv")
FC = json.loads((PROC / "feature_columns.json").read_text())
S = len(shot_df)

# "complete seasons" = the four 2015/16 league seasons
COMPLETE = shot_df["subset_1516"].astype(bool).values
# focus club: only defined for single-club seasons in which one team plays every match
IS_FOCUS = shot_df["is_focus_team"].astype(bool).values
season_has_focus = shot_df.groupby(["league", "season"])["is_focus_team"].transform("max").astype(bool).values
FOCUS_DEFINED = (~COMPLETE) & season_has_focus

print(f"shots: {shot_df.shape} | player rows (incl. shooters): {player_df.shape}")
print("complete league seasons used for validation / test / headline numbers:")
print(shot_df[COMPLETE].groupby(["league", "season"]).size().rename("shots").to_frame().T.to_string())

In [ ]:
# ---- node table: the shooter row followed by the visible players, for every shot ----------------
NODE_BASIC = FC["player_basic"]          # position, relation to shooter and goal, team, keeper, shooter, role
NODE_ENG = FC["player_engineered"]       # dist_to_shot_line, in_cone, cone_clearance, goal_blocked (from 01-Data)
NODE_COLS = NODE_BASIC + NODE_ENG

shot_pos = pd.Series(np.arange(S), index=shot_df["shot_id"].values)
player_df["_s"] = player_df["shot_id"].map(shot_pos)
assert player_df["_s"].notna().all(), "player_table has shots that are not in shot_table"
player_df["_s"] = player_df["_s"].astype(int)
player_df = player_df.sort_values(["_s", "player_idx"], kind="stable").reset_index(drop=True)

NCOUNT = np.bincount(player_df["_s"].values, minlength=S)               # visible players + the shooter
NODE_PTR = np.concatenate([[0], np.cumsum(NCOUNT)])
assert (player_df.groupby("_s")["is_shooter"].sum() == 1).all(), "every shot needs exactly one shooter row"
assert (player_df["is_shooter"].values[NODE_PTR[:-1]] == 1).all(), "the shooter must be the first row of each shot"

vals = player_df[NODE_COLS].values.astype("float32")
POS = player_df[["x", "y"]].values.astype("float32")                     # yards, used to build edges (not a feature)
TEAM, KEEPER, SHOOTER = (player_df[c].values.astype(int) for c in ["teammate", "is_keeper", "is_shooter"])

def standardise(a, train_mask):
    a = a.copy()
    for j in range(a.shape[1]):
        if set(np.unique(a[:, j])) <= {0.0, 1.0}:
            continue
        mu, sd = a[train_mask, j].mean(), a[train_mask, j].std()
        a[:, j] = (a[:, j] - mu) / max(sd, 1e-6)
    return a

is_train_shot = (shot_df["split"] == "train").values
is_train_node = np.repeat(is_train_shot, NCOUNT)
X_FEATS = {"basic": torch.tensor(standardise(vals[:, :len(NODE_BASIC)], is_train_node)),
           "engineered": torch.tensor(standardise(vals, is_train_node))}

SHOT_FEATURES = FC["shot_features"]
U = torch.tensor(standardise(shot_df[SHOT_FEATURES].values.astype("float32"), is_train_shot))
Y = torch.tensor(shot_df["is_goal"].values.astype("float32"))
MATCH_IDS, BENCH = shot_df["match_id"].values, shot_df["statsbomb_xg"].values

split = shot_df["split"].values
IDX = {"train_all": np.flatnonzero(split == "train"), "train_cs": np.flatnonzero((split == "train") & COMPLETE),
       "val": np.flatnonzero(split == "val"), "test": np.flatnonzero(split == "test")}
if MAX_TRAIN:   # smoke mode only
    rng0 = np.random.default_rng(0)
    for k in ["train_all", "train_cs"]:
        IDX[k] = np.sort(rng0.choice(IDX[k], min(MAX_TRAIN, len(IDX[k])), replace=False))
IN_CS = {s: COMPLETE[IDX[s]] for s in ["val", "test"]}            # complete-season mask inside val / test

overview = pd.DataFrame({
    name: {"shots": len(i), "matches": len(np.unique(MATCH_IDS[i])), "goal rate": Y[i].mean().item()}
    for name, i in [("train (all seasons)", IDX["train_all"]), ("train (complete seasons only)", IDX["train_cs"]),
                    ("val, complete seasons", IDX["val"][IN_CS["val"]]), ("val, single-club", IDX["val"][~IN_CS["val"]]),
                    ("test, complete seasons", IDX["test"][IN_CS["test"]]), ("test, single-club", IDX["test"][~IN_CS["test"]])]}).T
print(overview.round(3))
print(f"\nnodes per shot (incl. shooter): mean {NCOUNT.mean():.1f}, max {NCOUNT.max()}")
print(f"node features: {len(NODE_BASIC)} basic, {len(NODE_COLS)} with engineered | shot features: {len(SHOT_FEATURES)}")

Single-club seasons have a noticeably higher goal rate than the complete seasons. This is the first hint that they
are not a representative sample, and the reason the headline numbers use complete-season matches.

## 2. How a shot becomes a graph

**Nodes:** the shooter and every visible player. Each node carries the features above.

**Edges** say which players exchange information. We compare six rules (Fig 3):

| rule | edges | idea |
|---|---|---|
| `none` | no edges | **set model**: each player is processed on its own, then pooled |
| `star` | shooter ↔ players in the region (+ opposition keeper) | only the players near the shot talk to the shooter |
| `star_keeper` | `star` + keeper ↔ region players | also lets the keeper "see" the players in front of them |
| `region` | every pair within {shooter, region players, keeper} | the local defensive structure is fully connected |
| `region_noshooter` | every pair within {region players, keeper}, **no edge touches the shooter** | tests relations *between defenders and keeper* only (see note) |
| `full` | every pair of players | no football knowledge in the edges; the model decides what matters |

**Region** around the shooter: a half-disc of radius `r_front` in front (towards goal), a half-disc of radius
`r_back` behind, plus the **shooting cone** (triangle shooter → posts). Default 7 yd / 4 yd. Players outside the
region are still nodes (so still in the pooled summary), they just do not send or receive messages.

**Edge features** for each edge: distance, `dx`, `dy` (all / 20 yd) and a same-team flag.

**Note on what edges can add.** Every player node already carries its offset and distance *to the shooter*
(`dx_to_shooter`, `dy_to_shooter`, `dist_to_shooter`). So the edge features of a shooter–player edge repeat information the set
model already has. Edges only add *new* information when they connect two players who are not the shooter
(keeper ↔ defender, defender ↔ defender). `region_noshooter` isolates exactly that.

In [ ]:
def _cross(ax, ay, bx, by, px, py):
    return (bx - ax) * (py - ay) - (by - ay) * (px - ax)

def in_cone(px, py, sx, sy):
    d1, d2, d3 = (_cross(sx, sy, GOAL_X, POST_L, px, py), _cross(GOAL_X, POST_L, GOAL_X, POST_R, px, py),
                  _cross(GOAL_X, POST_R, sx, sy, px, py))
    return ~(((d1 < 0) | (d2 < 0) | (d3 < 0)) & ((d1 > 0) | (d2 > 0) | (d3 > 0)))

def region_mask(px, py, sx, sy, r_front, r_back):
    dx, d = px - sx, np.hypot(px - sx, py - sy)
    return np.where(dx > 0, d <= r_front, d <= r_back) | in_cone(px, py, sx, sy)

def _pairs(nodes):
    a, b = np.meshgrid(nodes, nodes, indexing="ij")
    m = a != b
    return a[m], b[m]

RULES = ["none", "star", "star_keeper", "region", "region_noshooter", "full"]
REGION_RULES = ["star", "star_keeper", "region", "region_noshooter"]

def build_edges(rule, px, py, team, keeper, shooter, r_front, r_back):
    # Directed edges (src, dst) as local node indices for ONE shot.
    n = len(px)
    if rule == "none":
        return np.zeros(0, int), np.zeros(0, int)
    if rule == "full":
        return _pairs(np.arange(n))
    s = int(np.flatnonzero(shooter)[0])
    keepers = np.flatnonzero((keeper == 1) & (team == 0))
    reg = region_mask(px, py, px[s], py[s], r_front, r_back); reg[s] = False
    members = np.setdiff1d(np.union1d(np.flatnonzero(reg), keepers), [s])
    if rule == "region":
        return _pairs(np.concatenate([[s], members]))
    if rule == "region_noshooter":
        return _pairs(members)
    src, dst = [np.full(len(members), s), members], [members, np.full(len(members), s)]
    if rule == "star_keeper":
        for k in keepers:
            others = members[members != k]
            src += [np.full(len(others), k), others]; dst += [others, np.full(len(others), k)]
    return np.concatenate(src), np.concatenate(dst)

In [ ]:
def _ranges(starts, counts):
    # concatenate arange(start, start + count) for many pairs, vectorised
    offs = np.repeat(np.cumsum(counts) - counts, counts)
    return np.repeat(starts, counts) + (np.arange(int(counts.sum())) - offs)

class Batch:
    def to(self, device):
        for k, v in self.__dict__.items():
            if torch.is_tensor(v):
                setattr(self, k, v.to(device))
        return self

class GraphStore:
    # Edges for every shot under one rule, stored flat. batch() glues shots into one disconnected graph.
    def __init__(self, src, dst, ecount):
        self.src, self.dst, self.ecount = src, dst, ecount
        self.eptr = np.concatenate([[0], np.cumsum(ecount)])
        self.mean_edges = float(ecount.mean())

    def batch(self, idx, feats):
        idx = np.asarray(idx)
        nc, ec = NCOUNT[idx], self.ecount[idx]
        nrows = _ranges(NODE_PTR[idx], nc)
        erows = _ranges(self.eptr[idx], ec)
        noff = np.repeat(np.cumsum(nc) - nc, ec)
        b = Batch()
        b.x = X_FEATS[feats][nrows]
        b.is_shooter = torch.from_numpy(SHOOTER[nrows] == 1)
        b.node_graph = torch.from_numpy(np.repeat(np.arange(len(idx)), nc))
        b.src = torch.from_numpy(self.src[erows].astype(np.int64) + noff)
        b.dst = torch.from_numpy(self.dst[erows].astype(np.int64) + noff)
        pos, team = torch.from_numpy(POS[nrows]), torch.from_numpy(TEAM[nrows])
        d = pos[b.src] - pos[b.dst]
        b.edge_attr = torch.cat([d.norm(dim=1, keepdim=True) / 20, d / 20,
                                 (team[b.src] == team[b.dst]).float().unsqueeze(1)], 1)
        b.n_graphs, b.u, b.y = len(idx), U[idx], Y[idx]
        return b

STORES, BUILD_TIME = {}, {}
def get_store(rule, r_front=7.0, r_back=4.0):
    key = (rule, r_front, r_back) if rule in REGION_RULES else (rule,)
    if key not in STORES:
        t0 = time.time()
        srcs, dsts, ecount = [], [], np.zeros(S, int)
        for i in range(S):
            a, b = NODE_PTR[i], NODE_PTR[i + 1]
            es, ed = build_edges(rule, POS[a:b, 0], POS[a:b, 1], TEAM[a:b], KEEPER[a:b], SHOOTER[a:b], r_front, r_back)
            srcs.append(es.astype(np.int16)); dsts.append(ed.astype(np.int16)); ecount[i] = len(es)
        STORES[key] = GraphStore(np.concatenate(srcs), np.concatenate(dsts), ecount)
        BUILD_TIME[key] = time.time() - t0                       # one-off cost of building every shot's edges (used in 7.5)
        print(f"  built {key}: {STORES[key].mean_edges:.1f} directed edges per shot ({BUILD_TIME[key]:.0f}s)")
    return STORES[key]

In [ ]:
def draw_pitch_shot(ax, i, rule, r_front=7.0, r_back=4.0, node_size=None, title=None):
    a, b = NODE_PTR[i], NODE_PTR[i + 1]
    px, py = POS[a:b, 0], POS[a:b, 1]
    team, kp, sh = TEAM[a:b], KEEPER[a:b], SHOOTER[a:b]
    s = int(np.flatnonzero(sh)[0]); sx, sy = px[s], py[s]
    ax.add_patch(Polygon([[sx, sy], [GOAL_X, POST_L], [GOAL_X, POST_R]], alpha=0.12, color="tab:green"))
    if rule in REGION_RULES:
        ax.add_patch(Wedge((sx, sy), r_front, -90, 90, alpha=0.10, color="tab:blue"))
        ax.add_patch(Wedge((sx, sy), r_back, 90, 270, alpha=0.10, color="tab:orange"))
    es, ed = build_edges(rule, px, py, team, kp, sh, r_front, r_back)
    for u_, v_ in zip(es, ed):
        ax.plot([px[u_], px[v_]], [py[u_], py[v_]], color="grey", lw=0.6, alpha=0.6, zorder=1)
    col_ = np.where(sh == 1, "gold", np.where(team == 1, "tab:blue", "tab:red"))
    size = node_size if node_size is not None else np.where(kp == 1, 90, 45)
    ax.scatter(px, py, c=col_, s=size, edgecolor="k", zorder=3)
    ax.plot([GOAL_X, GOAL_X], [POST_L, POST_R], color="k", lw=4)
    ax.set_title(title or f"{rule}: {len(es)} edges", fontsize=9); ax.set_aspect("equal")
    ax.set_xlim(sx - 22, 122); ax.set_ylim(sy - 22, sy + 22); ax.tick_params(labelsize=7)

pool = IDX["val"][IN_CS["val"]] if IN_CS["val"].any() else IDX["val"]
cands = [i for i in pool if NCOUNT[i] >= 14 and POS[NODE_PTR[i], 0] > 100]
EXAMPLE = int(cands[0] if cands else pool[0])
fig, axes = plt.subplots(1, 6, figsize=(21, 4.2), sharey=True)
for ax, rule in zip(axes, RULES):
    draw_pitch_shot(ax, EXAMPLE, rule, title=f"{'set model' if rule == 'none' else rule}")
fig.suptitle("Fig 3. One shot under each edge rule. Gold = shooter, blue = teammates, red = opponents (large = keeper); "
             "green = shooting cone, blue/orange = front/back half-discs", fontsize=9)
plt.tight_layout(); plt.savefig(FIGURES / "fig03_edge_rules.png", dpi=150); plt.show()

In [ ]:
edge_counts = pd.DataFrame({r: {"mean edges per shot": get_store(r).mean_edges,
                                "max edges per shot": int(get_store(r).ecount.max())} for r in RULES}).T
edge_counts.round(1)

The number of edges controls the cost of a graph model (Section 7). `full` grows like n² in the number of players,
the region rules grow roughly like n.

## 3. The models

Both models share one code path, so the only difference between them is message passing.

1. **Encode** each node (players and shooter): `h_i = MLP(x_i)`.
2. **Message passing** (graph model only, 2 rounds). For each edge j → i: `m_ji = MLP([h_i, h_j, e_ji])`;
   each node combines its incoming messages with the aggregator `a_i = AGG_j(m_ji)` and updates
   `h_i ← LayerNorm(h_i + MLP([h_i, a_i]))`. Two rounds let information travel two steps,
   e.g. defender → defender → shooter.
3. **Readout**: the shooter's vector, concatenated with a pooled summary of all nodes `AGG_i(h_i)`, concatenated with
   the shot features (distance, angle, body part, ...). An MLP gives the logit of P(goal).

With 0 rounds this is a **Deep Sets** model: each node is encoded independently and pooled, so the model sees
every player but never how players relate to each other. That is our set model.
The model uses **LayerNorm** (per node), not BatchNorm, so one shot's prediction never depends on the other shots in its
batch. This matters for data-parallel training (Section 7.4).

**Aggregators** (`AGG`), i.e. how a group of players or messages is summarised:
* `sum`: adds everything up. Sensitive to *how many* players there are (a crowded box gives a big signal).
* `mean`: the average. Ignores the count, describes the "typical" player.
* `max`: the strongest signal per feature. Close to "what does the single most dangerous defender do".
* `attn`: a learned weighted average. The model scores each player and the weights sum to 1, so it can learn
  that a defender in the shooting lane matters more than one 30 yards away.

The same aggregator is used for message passing and for the readout, so the set and graph models are tuned
on equal terms.

In [ ]:
def scatter_sum(src, index, n):
    return src.new_zeros((n,) + src.shape[1:]).index_add_(0, index, src)

def scatter_mean(src, index, n):
    cnt = torch.bincount(index, minlength=n).clamp(min=1).to(src.dtype).unsqueeze(1)
    return scatter_sum(src, index, n) / cnt

def scatter_max(src, index, n):
    out = src.new_full((n,) + src.shape[1:], 0.0)
    return out.scatter_reduce(0, index.unsqueeze(1).expand_as(src), src, reduce="amax", include_self=False)

def segment_softmax(score, index, n):
    mx = score.new_full((n,), float("-inf")).scatter_reduce(0, index, score, reduce="amax", include_self=True)
    ex = torch.exp(score - mx[index])
    return ex / (scatter_sum(ex, index, n)[index] + 1e-12)

def aggregate(x, index, n, how, scorer=None):
    if how == "sum":
        return scatter_sum(x, index, n), None
    if how == "mean":
        return scatter_mean(x, index, n), None
    if how == "max":
        return scatter_max(x, index, n), None
    alpha = segment_softmax(scorer(x).squeeze(1), index, n)            # attention
    return scatter_sum(alpha.unsqueeze(1) * x, index, n), alpha

class MPLayer(nn.Module):
    def __init__(self, h, e_dim, aggr):
        super().__init__()
        self.aggr = aggr
        self.msg = nn.Sequential(nn.Linear(2 * h + e_dim, h), nn.ReLU(), nn.Linear(h, h))
        self.upd = nn.Sequential(nn.Linear(2 * h, h), nn.ReLU(), nn.Linear(h, h))
        self.norm = nn.LayerNorm(h)
        self.scorer = nn.Linear(h, 1) if aggr == "attn" else None

    def forward(self, h, src, dst, e):
        if src.numel() == 0:
            agg = torch.zeros_like(h)
        else:
            m = self.msg(torch.cat([h[dst], h[src], e], 1))
            agg, _ = aggregate(m, dst, h.size(0), self.aggr, self.scorer)
        return self.norm(h + self.upd(torch.cat([h, agg], 1)))

class XGNet(nn.Module):
    # rounds = 0 -> set model (Deep Sets); rounds > 0 -> graph model (message passing)
    def __init__(self, node_dim, shot_dim, rounds=2, aggr="sum", hidden=64, edge_dim=4, dropout=0.1):
        super().__init__()
        self.aggr = aggr
        self.enc = nn.Sequential(nn.Linear(node_dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
        self.layers = nn.ModuleList([MPLayer(hidden, edge_dim, aggr) for _ in range(rounds)])
        self.pool_scorer = nn.Linear(hidden, 1) if aggr == "attn" else None
        self.head = nn.Sequential(nn.Linear(2 * hidden + shot_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
                                  nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, 1))
        self.last_alpha = None

    def forward(self, b):
        h = self.enc(b.x)
        for layer in self.layers:
            h = layer(h, b.src, b.dst, b.edge_attr)
        pooled, self.last_alpha = aggregate(h, b.node_graph, b.n_graphs, self.aggr, self.pool_scorer)
        return self.head(torch.cat([h[b.is_shooter], pooled, b.u], 1)).squeeze(1)

n_params = lambda m: sum(p.numel() for p in m.parameters())
for feats in ["basic", "engineered"]:
    d = X_FEATS[feats].shape[1]
    print(f"{feats:>10} features ({d}): set model {n_params(XGNet(d, U.shape[1], rounds=0)):,} parameters, "
          f"graph model {n_params(XGNet(d, U.shape[1], rounds=2)):,} parameters")

The graph model has more parameters than the set model (the message and update MLPs). This is reported next to
every result so that a graph win cannot simply be put down to size without checking.

### Correctness tests
* **Permutation invariance:** shuffling the order of players inside a shot must not change the prediction.
* **Batching:** a shot must get the same prediction alone or inside a batch.

In [ ]:
def check_model():
    store, idx = get_store("full"), IDX["val"][:8]
    for aggr in ["sum", "mean", "max", "attn"]:
        torch.manual_seed(0)
        model = XGNet(X_FEATS["basic"].shape[1], U.shape[1], rounds=2, aggr=aggr).eval()
        with torch.no_grad():
            b = store.batch(idx, "basic")
            together = model(b)
            alone = torch.cat([model(store.batch(idx[i:i + 1], "basic")) for i in range(len(idx))])
            nc = NCOUNT[idx]
            perm = torch.cat([torch.randperm(int(c)) + int(o) for c, o in zip(nc, np.cumsum(nc) - nc)])
            inv = torch.empty_like(perm); inv[perm] = torch.arange(len(perm))
            b2 = copy.copy(b)
            b2.x, b2.is_shooter, b2.node_graph = b.x[perm], b.is_shooter[perm], b.node_graph[perm]
            b2.src, b2.dst = inv[b.src], inv[b.dst]
            shuffled = model(b2)
        assert torch.allclose(together, alone, atol=1e-4), f"{aggr}: batch vs single mismatch"
        assert torch.allclose(together, shuffled, atol=1e-4), f"{aggr}: not permutation invariant"
    print("OK: predictions do not depend on player order or on batching (sum / mean / max / attn).")

check_model()

## 4. Training, evaluation and how we deal with noise

* **Loss:** binary cross-entropy on the unweighted data. No resampling of goals, because that would distort the
  probabilities and calibration is what makes an xG model useful.
* **Optimiser:** AdamW (lr 1e-3, weight decay 1e-4), batch size 256, gradient clipping.
  **Early stopping** on validation log loss over complete-season matches (patience 4), best weights restored.
* **Metrics:**
  * **log loss** (main metric): heavily penalises confident wrong predictions. Lower is better.
  * **Brier score**: mean squared error of the probability. Lower is better.
  * **ROC AUC**: how well the model ranks goals above non-goals (ignores calibration). Higher is better.
* **Fairness rule:** any choice that applies to both models (aggregator, features, training data) is tuned for both.
  Graph-only choices (edge rule, region size) are tuned for the graph model only.
* **StatsBomb xG** is shown as an external benchmark, not as a ceiling and not as a like-for-like opponent: it is a different
  model, trained on different data, and we use its published values.

### Noise: what counts as a difference?
Differences between configurations are often in the third decimal place, so we need to know how large the noise is.
There are three sources.

1. **Training seed.** Every configuration, in every experiment, is trained with several seeds. Tables show the mean and the
   standard deviation over seeds.
2. **Which matches are in the validation set.** For two configurations we take the per-shot log-loss difference (averaged over
   seeds) and **resample whole matches** with replacement (shots in a match are not independent) to get a 95% interval
   for the mean difference. Two configurations are *distinguishable* only if that interval excludes 0.
3. **Selection bias.** The best of many noisy comparisons is partly lucky. So within each model we keep the **cheapest**
   configuration (fewest edges × training shots) among those that are *not distinguishable from the best*, and only
   fall back on the lowest log loss to break ties. Where the data cannot separate two settings, we do not pretend they differ.

For the final test we also ask whether the two models are *equivalent*: the 95% interval of the difference lies inside
±`EQUIV_MARGIN`. That is a stronger statement than "no clear difference", which could just mean the test is too small.

In [ ]:
def metrics(y, p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return {"logloss": log_loss(y, p, labels=[0, 1]), "brier": brier_score_loss(y, p), "auc": roc_auc_score(y, p)}

def shot_loss(y, p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))

def boot_diff(match_ids, d, n_boot=None, seed=0):
    # d = per-shot loss difference (A - B). Resamples whole matches. Returns mean, 95% interval, bootstrap draws.
    n_boot = n_boot or N_BOOT
    g = pd.DataFrame({"m": match_ids, "d": d}).groupby("m")["d"].agg(["sum", "count"])
    s, c = g["sum"].values, g["count"].values
    idx = np.random.default_rng(seed).integers(0, len(s), (n_boot, len(s)))
    draws = s[idx].sum(1) / c[idx].sum(1)
    return float(np.mean(d)), np.percentile(draws, [2.5, 97.5]), draws

@torch.no_grad()
def predict(model, store, idx, feats, bs=2048):
    model.eval()
    return torch.cat([torch.sigmoid(model(store.batch(idx[i:i + bs], feats).to(DEVICE))).cpu()
                      for i in range(0, len(idx), bs)]).numpy()

BASE = dict(rule="region", r_front=7.0, r_back=4.0, aggr="sum", feats="basic", train="all", frac=1.0)

def train_indices(cfg, seed=0):
    idx = IDX["train_cs"] if cfg["train"] == "cs" else IDX["train_all"]
    if cfg["train"] == "all_sub":     # all seasons, subsampled (by match) to the size of the complete-season training set
        n_target = len(np.unique(MATCH_IDS[IDX["train_cs"]]))
        keep = np.random.default_rng(123).choice(np.unique(MATCH_IDS[idx]), n_target, replace=False)
        idx = idx[np.isin(MATCH_IDS[idx], keep)]
    if cfg["frac"] < 1.0:             # learning curve: a fraction of the training matches
        m = np.unique(MATCH_IDS[idx])
        keep = np.random.default_rng(1000 + seed).choice(m, max(1, int(cfg["frac"] * len(m))), replace=False)
        idx = idx[np.isin(MATCH_IDS[idx], keep)]
    return idx

VAL_CS = IDX["val"][IN_CS["val"]]
Y_VAL_CS, M_VAL_CS = Y[VAL_CS].numpy(), MATCH_IDS[VAL_CS]

def train_model(cfg, seed):
    rounds = 0 if cfg["rule"] == "none" else 2
    store = get_store(cfg["rule"], cfg["r_front"], cfg["r_back"])
    tr = train_indices(cfg, seed)
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = XGNet(X_FEATS[cfg["feats"]].shape[1], U.shape[1], rounds=rounds, aggr=cfg["aggr"]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    lossf = nn.BCEWithLogitsLoss()
    best, best_state, best_ep, bad, ep_times = np.inf, None, 0, 0, []
    for ep in range(1, MAX_EPOCHS + 1):
        model.train(); t0 = time.time()
        order = rng.permutation(tr)
        for i in range(0, len(order), 256):
            b = store.batch(order[i:i + 256], cfg["feats"]).to(DEVICE)
            opt.zero_grad()
            lossf(model(b), b.y).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        ep_times.append(time.time() - t0)
        v = metrics(Y_VAL_CS, predict(model, store, VAL_CS, cfg["feats"]))["logloss"]
        if v < best - 1e-5:
            best, best_state, best_ep, bad = v, copy.deepcopy(model.state_dict()), ep, 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    preds = {s: predict(model, store, IDX[s], cfg["feats"]) for s in ["val", "test"]}
    info = dict(best_epoch=best_ep, epochs_run=ep, epoch_time_s=float(np.mean(ep_times)),
                n_train=len(tr), edges_per_shot=store.mean_edges, n_params=n_params(model))
    return model, info, preds

In [ ]:
RESULTS, PREDS, MODELS, RESULT_BY_KEY = [], {}, {}, {}
ckey = lambda cfg: tuple(sorted(cfg.items()))

def run(experiment, named_cfgs, seeds=None):
    # Train every configuration with every seed (reusing identical runs already trained) and tag them with the experiment.
    seeds = seeds or SEEDS
    for name, cfg in named_cfgs.items():
        for seed in seeds:
            k = (ckey(cfg), seed)
            if k not in PREDS:
                t0 = time.time()
                MODELS[k], info, PREDS[k] = train_model(cfg, seed)
                print(f"  {name:<36} seed {seed}: val log loss "
                      f"{metrics(Y_VAL_CS, PREDS[k]['val'][IN_CS['val']])['logloss']:.4f}  ({time.time() - t0:.0f}s)")
                RESULT_BY_KEY[k] = dict(key=k, **cfg, seed=seed, **info, used_in=set())
                RESULTS.append(RESULT_BY_KEY[k])
            RESULT_BY_KEY[k]["used_in"].add((experiment, name))

def exp_table(experiment):
    # One row per configuration: mean and sd over seeds, and the difference to the best configuration with a match-bootstrap interval.
    groups = {}
    for r in RESULTS:
        for exp, name in r["used_in"]:
            if exp == experiment:
                groups.setdefault(name, []).append(r)
    rows = []
    for name, rs in groups.items():
        cfg = dict(rs[0]["key"][0])
        p_val = [PREDS[r["key"]]["val"][IN_CS["val"]] for r in rs]
        ms = [metrics(Y_VAL_CS, p) for p in p_val]
        rows.append(dict(
            name=name, model="set" if cfg["rule"] == "none" else "graph",
            val_logloss=np.mean([m["logloss"] for m in ms]), seed_sd=np.std([m["logloss"] for m in ms], ddof=1) if len(ms) > 1 else np.nan,
            val_brier=np.mean([m["brier"] for m in ms]), val_auc=np.mean([m["auc"] for m in ms]),
            edges_per_shot=rs[0]["edges_per_shot"], epoch_time_s=np.mean([r["epoch_time_s"] for r in rs]),
            n_params=rs[0]["n_params"], best_epoch=np.mean([r["best_epoch"] for r in rs]), n_seeds=len(rs),
            cost=rs[0]["n_train"] * (1 + rs[0]["edges_per_shot"]),
            _loss=np.mean([shot_loss(Y_VAL_CS, p) for p in p_val], axis=0), _cfg=cfg))
    df = pd.DataFrame(rows)
    best = df.loc[df["val_logloss"].idxmin()]
    ds = []
    for _, r in df.iterrows():
        m, (lo, hi), _ = boot_diff(M_VAL_CS, r["_loss"] - best["_loss"])
        ds.append((m, lo, hi, (hi < 0 or lo > 0) if r["name"] != best["name"] else False))
    df[["d_vs_best", "ci_low", "ci_high", "distinguishable"]] = pd.DataFrame(ds, index=df.index)
    return df.sort_values("val_logloss").reset_index(drop=True)

PUBLIC = ["name", "model", "val_logloss", "seed_sd", "d_vs_best", "ci_low", "ci_high", "distinguishable",
          "val_brier", "val_auc", "edges_per_shot", "epoch_time_s", "n_params", "n_seeds"]
def show(experiment):
    return exp_table(experiment)[PUBLIC].round(4)

CHOICES = {}
def choose(experiment, model, allow=lambda cfg: True):
    # Cheapest configuration among those not distinguishable from the best (by mean) of this model.
    # `allow` removes configurations that are diagnostic controls rather than real candidates.
    df = exp_table(experiment)
    df = df[(df["model"] == model) & df["_cfg"].apply(allow)].reset_index(drop=True)
    best = df.loc[df["val_logloss"].idxmin()]
    tied = []
    for _, r in df.iterrows():
        m, (lo, hi), _ = boot_diff(M_VAL_CS, r["_loss"] - best["_loss"])
        if r["name"] == best["name"] or not (hi < 0 or lo > 0):
            tied.append(r)
    chosen = min(tied, key=lambda r: (r["cost"], r["val_logloss"]))
    CHOICES[(experiment, model)] = dict(best=best["name"], tied=[r["name"] for r in tied], chosen=chosen["name"], n=len(df))
    print(f"  [{experiment}, {model}] best by mean: {best['name']} | not distinguishable from it: {len(tied)} of {len(df)}"
          f" | kept (cheapest of those): {chosen['name']}")
    return dict(chosen["_cfg"])

SB_VAL = metrics(Y_VAL_CS, BENCH[VAL_CS])["logloss"]

def plot_experiment(experiment, title, fig_name):
    s = exp_table(experiment)
    fig, ax = plt.subplots(figsize=(7.5, 0.42 * len(s) + 1.5))
    colours = s["model"].map({"set": "tab:orange", "graph": "tab:blue"})
    ax.barh(s["name"], s["val_logloss"], xerr=s["seed_sd"].fillna(0), color=colours, capsize=2)
    sd = s["seed_sd"].fillna(0)
    lo, hi = min(s["val_logloss"].min(), SB_VAL), (s["val_logloss"] + sd).max()
    span = max(hi - lo, 1e-3)
    ax.set_xlim(lo - 0.25 * span, hi + 0.45 * span)           # room on the right for the labels
    ax.axvline(SB_VAL, color="k", ls="--", lw=1)
    for y_, (v, e, dist) in enumerate(zip(s["val_logloss"], sd, s["distinguishable"])):
        tag = "  (best)" if y_ == 0 else ("" if dist else "  (≈ best)")
        ax.text(v + e + 0.03 * span, y_, f"{v:.4f}{tag}", va="center", fontsize=8)   # placed past the error bar
    ax.invert_yaxis(); ax.set_xlabel("validation log loss, complete-season matches (lower is better)")
    ax.set_title(title, fontsize=10)
    ax.scatter([], [], c="tab:orange", marker="s", label="set"); ax.scatter([], [], c="tab:blue", marker="s", label="graph")
    ax.plot([], [], "k--", lw=1, label=f"StatsBomb xG ({SB_VAL:.4f})")
    fig.legend(loc="lower center", ncol=3, fontsize=8, frameon=False)
    plt.tight_layout(rect=(0, 0.07, 1, 1)); plt.savefig(FIGURES / fig_name, dpi=150); plt.show()

## 5. Experiments (tuning on the validation set)

Each experiment changes **one** thing and keeps the best setting so far by the rule in Section 4
(a greedy chain: edge rule → region size → aggregation → features → training data). This is much cheaper than trying every
combination, but it can miss interactions between settings. The test set is not touched until Section 6.

How to read the figures: bars are mean validation log loss on complete-season matches (lower is better), error bars are the
standard deviation over seeds, orange = set model, blue = graph model, dashed line = StatsBomb xG.
`(≈ best)` marks a configuration whose difference to the best has a 95% interval that includes 0. In the tables,
`d_vs_best` is the mean difference to the best configuration in that experiment, `ci_low`/`ci_high` its 95% match-bootstrap interval,
and `distinguishable` says whether that interval excludes 0.

### 5.1 Edge rule: does connecting players help, and which connections?
The set model is the reference. If any graph rule beats it, relations between players carry information that
the individual player features do not.

In [ ]:
E1 = {"set model": {**BASE, "rule": "none"},
      "graph: star": {**BASE, "rule": "star"},
      "graph: star + keeper": {**BASE, "rule": "star_keeper"},
      "graph: region": {**BASE, "rule": "region"},
      "graph: region, no shooter edges": {**BASE, "rule": "region_noshooter"},
      "graph: full": {**BASE, "rule": "full"}}
run("5.1 edge rule", E1)
display(show("5.1 edge rule"))
plot_experiment("5.1 edge rule", "Fig 4. Experiment 5.1: edge rule", "fig04_exp_edge_rule.png")

# the central question: each graph rule against the set model
t51 = exp_table("5.1 edge rule")
ref = t51.loc[t51["name"] == "set model", "_loss"].iloc[0]
rows = []
for _, r in t51[t51["model"] == "graph"].iterrows():
    m, (lo, hi), _ = boot_diff(M_VAL_CS, r["_loss"] - ref)
    rows.append({"graph rule": r["name"], "graph − set": m, "ci_low": lo, "ci_high": hi,
                 "verdict": "graph better" if hi < 0 else ("set better" if lo > 0 else "no clear difference")})
VS_SET = pd.DataFrame(rows).set_index("graph rule")
display(VS_SET.round(4))

BEST = {"set": dict(E1["set model"]), "graph": choose("5.1 edge rule", "graph")}

`edges_per_shot` and `epoch_time_s` show the cost of each rule. These are used again in Section 7.

### 5.2 Region size: how far around the shooter should connections reach?
Graph-only, and only for the region-based rules. We try a small, the default and a large region on the rule chosen in 5.1.
Small regions give cheaper graphs; large ones move towards the `full` rule.

In [ ]:
rule = BEST["graph"]["rule"]
if rule in REGION_RULES:
    E2 = {f"{rule} small (5 / 3 yd)": {**BEST["graph"], "r_front": 5.0, "r_back": 3.0},
          f"{rule} default (7 / 4 yd)": {**BEST["graph"], "r_front": 7.0, "r_back": 4.0},
          f"{rule} large (10 / 6 yd)": {**BEST["graph"], "r_front": 10.0, "r_back": 6.0}}
    run("5.2 region size", E2)
    display(show("5.2 region size"))
    plot_experiment("5.2 region size", "Fig 5. Experiment 5.2: region size (front / back radius)", "fig05_exp_region.png")
    BEST["graph"] = choose("5.2 region size", "graph")
else:
    print(f"Edge rule '{rule}' has no region, so Experiment 5.2 does not apply.")
print("graph model so far:", BEST["graph"])

### 5.3 Aggregation: how should a group of players be summarised?
Applies to both models (fairness rule). For the set model it is how the players are pooled; for the graph model it is
also how each player combines its incoming messages. See Section 3 for what each aggregator means.

In [ ]:
E3 = {f"{m}: {a}": {**BEST[m], "aggr": a} for m in ["set", "graph"] for a in ["sum", "mean", "max", "attn"]}
run("5.3 aggregation", E3)
display(show("5.3 aggregation"))
plot_experiment("5.3 aggregation", "Fig 6. Experiment 5.3: aggregation", "fig06_exp_aggregation.png")
BEST = {m: choose("5.3 aggregation", m) for m in ["set", "graph"]}
print("set:", BEST["set"]["aggr"], "| graph:", BEST["graph"]["aggr"])

### 5.4 Engineered features: does hand-made geometry help, or can the network learn it?
Both models, with and without the four engineered player features from `01-Data` (distance to the shooter→goal line, inside the shooting cone, distance to the edge of the cone, and the share of the goal the player blocks). If the graph model gains less from them than the set model, it is
learning some of that geometry itself from the edges.

In [ ]:
E4 = {}
for m in ["set", "graph"]:
    E4[f"{m}: basic"] = {**BEST[m], "feats": "basic"}
    E4[f"{m}: basic + engineered"] = {**BEST[m], "feats": "engineered"}
run("5.4 features", E4)
display(show("5.4 features"))
plot_experiment("5.4 features", "Fig 7. Experiment 5.4: engineered player features", "fig07_exp_features.png")

t54 = exp_table("5.4 features").set_index("name")
FEAT_GAIN = {}
for m in ["set", "graph"]:
    d, (lo, hi), _ = boot_diff(M_VAL_CS, t54.loc[f"{m}: basic", "_loss"] - t54.loc[f"{m}: basic + engineered", "_loss"])
    FEAT_GAIN[m] = (d, lo, hi)
    print(f"{m}: log-loss gain from engineered features {d:+.4f}  (95% interval [{lo:+.4f}, {hi:+.4f}])")
BEST = {m: choose("5.4 features", m) for m in ["set", "graph"]}

### 5.5 Training data: more data vs biased data
Single-club seasons add shots, but every one of those matches contains the same, usually dominant, team. We train on three
sets and evaluate on the unbiased complete-season validation matches:

* **all seasons**: everything;
* **complete seasons only**;
* **all seasons, same size**: all seasons subsampled (by match) to the size of the complete-season set.

Comparing *complete seasons only* with *all seasons, same size* separates **which** matches we train on from **how many**.

In [ ]:
E5 = {}
for m in ["set", "graph"]:
    for name, t in [("all seasons", "all"), ("complete seasons only", "cs"), ("all seasons, same size", "all_sub")]:
        E5[f"{m}: {name}"] = {**BEST[m], "train": t}
run("5.5 training data", E5)
display(show("5.5 training data"))
plot_experiment("5.5 training data", "Fig 8. Experiment 5.5: training data", "fig08_exp_training_data.png")

t55 = exp_table("5.5 training data").set_index("name")
TRAIN_CMP = []
for m in ["set", "graph"]:
    for a, b, what in [("all seasons", "complete seasons only", "more data (all vs complete only)"),
                       ("all seasons, same size", "complete seasons only", "which matches (same size, all vs complete only)")]:
        d, (lo, hi), _ = boot_diff(M_VAL_CS, t55.loc[f"{m}: {a}", "_loss"] - t55.loc[f"{m}: {b}", "_loss"])
        TRAIN_CMP.append({"model": m, "comparison": f"{a} − {b}", "question": what, "diff": d, "ci_low": lo, "ci_high": hi,
                          "distinguishable": not (lo < 0 < hi)})
TRAIN_CMP = pd.DataFrame(TRAIN_CMP)
display(TRAIN_CMP.round(4))

**Do models under-predict the focus club?** On the single-club validation matches we compare actual goals with
the summed xG, separately for the focus club and its opponents (only for seasons where one club plays in every match).
A positive `goals − xG` means the team scored more than the model expected (finishing quality, or a bias the model has not learnt).
StatsBomb xG is the reference. Model predictions are averaged over seeds.

In [ ]:
val_idx = IDX["val"]
club = ~IN_CS["val"]
focus_v, defined_v = IS_FOCUS[val_idx], FOCUS_DEFINED[val_idx]
pm = lambda cfg: np.mean([PREDS[(ckey(cfg), s)]["val"] for s in SEEDS], axis=0)
rows = []
for group, mask in [("complete-season matches (all teams)", IN_CS["val"]),
                    ("single-club: focus club", club & defined_v & focus_v),
                    ("single-club: opponents", club & defined_v & ~focus_v)]:
    r = {"group": group, "shots": int(mask.sum()), "goals": int(Y[val_idx][mask].sum())}
    r["StatsBomb"] = r["goals"] - BENCH[val_idx][mask].sum()
    for m in ["set", "graph"]:
        for tname, t in [("all seasons", "all"), ("complete seasons", "cs")]:
            r[f"{m}, trained on {tname}"] = r["goals"] - pm({**BEST[m], "train": t})[mask].sum()
    rows.append(r)
over = pd.DataFrame(rows).set_index("group")
print("goals − xG (positive = scored more than expected):")
display(over.round(1))

ax = over.drop(columns=["shots", "goals"]).T.plot.barh(figsize=(8, 4.8))
ax.axvline(0, color="k", lw=1); ax.set_xlabel("goals − expected goals (validation)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=3, fontsize=8, frameon=False)
ax.set_title("Fig 9. Over-performance against xG by group", fontsize=10)
plt.tight_layout(); plt.savefig(FIGURES / "fig09_focus_club_overperformance.png", dpi=150); plt.show()

# "all seasons, same size" is a control that separates which matches from how many; it is not a candidate for the final model
BEST = {m: choose("5.5 training data", m, allow=lambda cfg: cfg["train"] != "all_sub") for m in ["set", "graph"]}

### 5.6 Chosen configurations
These two configurations go to the final test.

In [ ]:
chosen = pd.DataFrame(BEST).T[["rule", "r_front", "r_back", "aggr", "feats", "train"]]
chosen.loc["set", ["r_front", "r_back"]] = "–"     # region size does not apply to the set model
chosen.index.name = "model"
display(chosen)

EXPERIMENTS = ["5.1 edge rule", "5.2 region size", "5.3 aggregation", "5.4 features", "5.5 training data"]
pd.concat([exp_table(e)[PUBLIC].assign(experiment=e) for e in EXPERIMENTS if any(e == x for r in RESULTS for x, _ in r["used_in"])]
          ).to_csv(RESULTS_DIR / "experiments.csv", index=False)

## 6. Final test: set vs graph

The chosen set and graph configurations have already been trained with every seed (they are reused from the experiments, since the runs are identical)
and are evaluated **once** on the test matches.
* Headline: **test, complete seasons**, the held-out unbiased matches. Single-club test matches and all test matches are shown too.
* Each model's headline prediction is the average over its seeds (a small ensemble). Because an ensemble is a little better than
  a single network, the table also shows the **mean log loss of the single seeds**, which is the fairer number to put next to StatsBomb.
* **Paired bootstrap over matches:** resample test matches with replacement and recompute the difference in log loss
  (shots in the same match are not independent). If the 95% interval excludes 0, the difference is unlikely to be noise.
  If it lies inside ±`EQUIV_MARGIN`, the models are practically **equivalent**.

In [ ]:
FINAL = {"set": BEST["set"], "graph": BEST["graph"]}
run("6 final", {f"{m} (final)": cfg for m, cfg in FINAL.items()}, seeds=SEEDS)       # no new training: identical runs are reused

y_test, t_cs = Y[IDX["test"]].numpy(), IN_CS["test"]
P = {m: np.mean([PREDS[(ckey(cfg), s)]["test"] for s in SEEDS], axis=0) for m, cfg in FINAL.items()}
P["StatsBomb xG"] = BENCH[IDX["test"]].astype(float)

table = []
for group, mask in [("test, complete seasons (headline)", t_cs), ("test, single-club", ~t_cs), ("test, all", np.ones_like(t_cs))]:
    for m, p in P.items():
        row = {"test set": group, "model": m, **metrics(y_test[mask], p[mask])}
        if m in FINAL:
            per_seed = [metrics(y_test[mask], PREDS[(ckey(FINAL[m]), s)]["test"][mask])["logloss"] for s in SEEDS]
            row["logloss, single seeds (mean)"] = np.mean(per_seed)
            row["logloss sd over seeds"] = np.std(per_seed, ddof=1) if len(per_seed) > 1 else np.nan
        table.append(row)
final = pd.DataFrame(table).set_index(["test set", "model"])
display(final.round(4))
final.to_csv(RESULTS_DIR / "final_test.csv")

In [ ]:
def verdict_of(a, b, lo, hi):
    if hi < 0:
        return f"{a} better"
    if lo > 0:
        return f"{b} better"
    if lo > -EQUIV_MARGIN and hi < EQUIV_MARGIN:
        return f"equivalent (interval inside ±{EQUIV_MARGIN})"
    return f"inconclusive (interval wider than ±{EQUIV_MARGIN})"

mt = MATCH_IDS[IDX["test"]][t_cs]
loss_e = {m: shot_loss(y_test[t_cs], p[t_cs]) for m, p in P.items()}                   # ensemble predictions
loss_s = {m: np.mean([shot_loss(y_test[t_cs], PREDS[(ckey(cfg), s)]["test"][t_cs]) for s in SEEDS], axis=0)
          for m, cfg in FINAL.items()}                                                   # single models, averaged over seeds

boot_rows, boots = [], {}
for label, a, b, la, lb in [("graph − set (ensembles)", "graph", "set", loss_e["graph"], loss_e["set"]),
                            ("graph − set (single models)", "graph", "set", loss_s["graph"], loss_s["set"]),
                            ("graph − StatsBomb xG (ensemble)", "graph", "StatsBomb xG", loss_e["graph"], loss_e["StatsBomb xG"]),
                            ("set − StatsBomb xG (ensemble)", "set", "StatsBomb xG", loss_e["set"], loss_e["StatsBomb xG"]),
                            ("graph − StatsBomb xG (single models)", "graph", "StatsBomb xG", loss_s["graph"], loss_e["StatsBomb xG"]),
                            ("set − StatsBomb xG (single models)", "set", "StatsBomb xG", loss_s["set"], loss_e["StatsBomb xG"])]:
    m, (lo, hi), boots[label] = boot_diff(mt, la - lb)
    boot_rows.append({"comparison": label, "log loss difference": m, "95% CI low": lo, "95% CI high": hi,
                      "verdict": verdict_of(a, b, lo, hi)})
boot_df = pd.DataFrame(boot_rows).set_index("comparison")
display(boot_df.round(4))
boot_df.to_csv(RESULTS_DIR / "final_bootstrap.csv")

**Fig 10** shows the things that matter for an xG model: how well it **ranks** shots (ROC curve, left), whether
its probabilities are **calibrated** (middle: within each band of predicted xG, the observed goal rate should equal the
predicted xG, i.e. points on the diagonal), and how large the set–graph difference is compared with the equivalence margin (right).

In [ ]:
def calib(y, p, bins=10):
    q = pd.qcut(p, bins, duplicates="drop")
    d = pd.DataFrame({"y": y, "p": p, "q": q}).groupby("q", observed=True).agg(p=("p", "mean"), y=("y", "mean"))
    return d["p"].values, d["y"].values

styles = {"set": ("tab:orange", "-"), "graph": ("tab:blue", "-"), "StatsBomb xG": ("k", "--")}
fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
for m, p in P.items():
    c, ls = styles[m]
    fpr, tpr, _ = roc_curve(y_test[t_cs], p[t_cs])
    ax[0].plot(fpr, tpr, color=c, ls=ls, label=f"{m} (AUC {roc_auc_score(y_test[t_cs], p[t_cs]):.3f})")
    cx, cy = calib(y_test[t_cs], p[t_cs])
    ax[1].plot(cx, cy, marker="o", color=c, ls=ls, label=m)
ax[0].plot([0, 1], [0, 1], ":", color="grey"); ax[0].set(xlabel="false positive rate", ylabel="true positive rate",
                                                       title="ROC curve (test, complete seasons)"); ax[0].legend(fontsize=8)
mx = ax[1].get_xlim()[1]
ax[1].plot([0, mx], [0, mx], ":", color="grey"); ax[1].set(xlabel="mean predicted xG (decile)", ylabel="observed goal rate",
                                                            title="Calibration (test, complete seasons)"); ax[1].legend(fontsize=8)
ax[2].hist(boots["graph − set (ensembles)"], bins=40, color="tab:blue", alpha=0.7)
ax[2].axvline(0, color="k", lw=1)
for v in (-EQUIV_MARGIN, EQUIV_MARGIN):
    ax[2].axvline(v, color="tab:red", ls="--", lw=1)
ax[2].set(xlabel="log loss(graph) − log loss(set)", ylabel="bootstrap samples",
          title="Bootstrap over test matches (red = equivalence margin)")
fig.suptitle("Fig 10. Final test on held-out complete-season matches", fontsize=10)
plt.tight_layout(); plt.savefig(FIGURES / "fig10_final_test.png", dpi=150); plt.show()

**Fig 11: where the models disagree.** The test shots (complete seasons) on which the set and graph models differ most. These show
what kind of situation the extra relational information changes.

In [ ]:
test_cs_idx = IDX["test"][t_cs]
diff = P["graph"][t_cs] - P["set"][t_cs]
picks = [np.argmax(diff), np.argmin(diff)]
graph_cfg = FINAL["graph"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for ax, j in zip(axes, picks):
    i = int(test_cs_idx[j])
    size = None
    if graph_cfg["aggr"] == "attn":   # show the graph model's readout attention as node size
        model = MODELS[(ckey(graph_cfg), SEEDS[0])].eval()
        with torch.no_grad():
            model(get_store(graph_cfg["rule"], graph_cfg["r_front"], graph_cfg["r_back"]).batch([i], graph_cfg["feats"]))
        size = 40 + 900 * model.last_alpha.numpy()
    draw_pitch_shot(ax, i, graph_cfg["rule"], graph_cfg["r_front"], graph_cfg["r_back"], node_size=size,
                    title=f"goal = {int(Y[i])} | set {P['set'][t_cs][j]:.2f}, graph {P['graph'][t_cs][j]:.2f}, "
                          f"StatsBomb {BENCH[i]:.2f}")
note = " Node size = graph model's attention weight." if graph_cfg["aggr"] == "attn" else ""
fig.suptitle("Fig 11. Test shots where the graph model is most above (left) and below (right) the set model." + note, fontsize=9)
plt.tight_layout(); plt.savefig(FIGURES / "fig11_disagreements.png", dpi=150); plt.show()

pd.DataFrame({"shot_id": shot_df["shot_id"].values[IDX["test"]], "match_id": MATCH_IDS[IDX["test"]],
              "complete_season": t_cs.astype(int), "is_goal": y_test,
              "p_set": P["set"], "p_graph": P["graph"], "p_statsbomb": P["StatsBomb xG"]}
             ).to_csv(RESULTS_DIR / "final_test_predictions.csv", index=False)

## 7. Scaling: what if there were 1000× more data?

No larger StatsBomb freeze-frame dataset is openly available. A realistic 1000× source would be *tracking* data
(all 22 players at 25 frames per second for every match), which is orders of magnitude bigger than one freeze frame per shot.
So we **measure** how cost and accuracy change with data volume on this machine, and combine the measurements with a small
cluster model whose assumptions are written out and can be changed. We keep the two clearly apart:

| Measured on this machine | Modelled (assumptions stated in the cell) |
|---|---|
| time per training step and where it goes (7.1) | wall-clock time with many workers (7.6) |
| accuracy against training size, and its extrapolation (7.2) | gradient all-reduce cost on a cluster (7.6) |
| time per epoch against data volume (7.3), against edges (7.4) | |
| speed-up from independent worker processes (7.5) | |
| data-parallel gradients equal single-process gradients (7.5, exact test) | |

Everything here is run on the two final configurations. If a GPU is available the timings use it; otherwise they are CPU timings,
and nothing below should be read as a GPU result.

### 7.1 Where does the time go in one training step?
A step is: build the batch (numpy), move it to the device, forward, backward, optimiser. Which of these parts can be shared out
across workers decides how the method scales.

In [ ]:
def sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

def make_model(cfg):
    return XGNet(X_FEATS[cfg["feats"]].shape[1], U.shape[1], rounds=0 if cfg["rule"] == "none" else 2, aggr=cfg["aggr"])

def profile_step(cfg, n_batches=None, bs=256):
    n_batches = n_batches or S_["profile_batches"]
    store = get_store(cfg["rule"], cfg["r_front"], cfg["r_back"])
    idx = train_indices(cfg)
    chunks = [idx[i:i + bs] for i in range(0, len(idx), bs)][:n_batches + 1]
    torch.manual_seed(0)
    model = make_model(cfg).to(DEVICE); model.train()
    opt, lossf = torch.optim.AdamW(model.parameters(), lr=1e-3), nn.BCEWithLogitsLoss()
    T = dict(batch_building=0.0, to_device=0.0, forward=0.0, backward=0.0, optimiser=0.0)
    for j, ids in enumerate(chunks):
        t0 = time.perf_counter(); b = store.batch(ids, cfg["feats"])
        t1 = time.perf_counter(); b = b.to(DEVICE); sync()
        t2 = time.perf_counter(); opt.zero_grad(); loss = lossf(model(b), b.y); sync()
        t3 = time.perf_counter(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); sync()
        t4 = time.perf_counter(); opt.step(); sync()
        t5 = time.perf_counter()
        if j > 0:                                                  # the first batch is a warm-up
            for k, dt in zip(T, [t1 - t0, t2 - t1, t3 - t2, t4 - t3, t5 - t4]):
                T[k] += dt
    return {k: 1000 * v / (len(chunks) - 1) for k, v in T.items()}      # milliseconds per step

PROFILE = pd.DataFrame({m: profile_step(cfg) for m, cfg in FINAL.items()}).T
PROFILE["total_ms"] = PROFILE.sum(axis=1)
PROFILE_PCT = PROFILE.drop(columns="total_ms").div(PROFILE["total_ms"], axis=0) * 100
STEP_S = (PROFILE["total_ms"] / 1000).to_dict()                         # seconds per step of 256 shots
display(PROFILE.round(2)); display(PROFILE_PCT.round(0).rename(columns=lambda c: c + " (%)"))

fig, ax = plt.subplots(figsize=(7.5, 2.6))
left = np.zeros(len(PROFILE_PCT))
for c in PROFILE_PCT.columns:
    ax.barh(PROFILE_PCT.index, PROFILE_PCT[c], left=left, label=c.replace("_", " ")); left += PROFILE_PCT[c].values
ax.set_xlabel("share of one training step (%)"); ax.legend(ncol=5, fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.35), frameon=False)
ax.set_title("Fig 12. Where the time of one training step goes (batch of 256 shots)", fontsize=10)
plt.tight_layout(); plt.savefig(FIGURES / "fig12_step_breakdown.png", dpi=150); plt.show()

### 7.2 Learning curve: what would 1000× more data buy in accuracy?
Cost is only half of scaling: it only makes sense to spend 1000× the compute if accuracy keeps improving. We train on 5% … 100% of
the training *matches* (every fraction with every seed) and fit a power law `loss(n) = a + b·n^(−c)`, where `a` is the loss that more data of
this kind could never get below with this model and these features. The fit has few points, so it is **indicative, not a forecast**.
The right-hand panel shows whether the set–graph gap shrinks as data grows, which would be the case for "the graph model needs more data".

In [ ]:
FRACS = S_["fracs"]
curve = []
for m, cfg in FINAL.items():
    for f in FRACS + [1.0]:
        c = {**cfg, "frac": f}
        if f < 1.0:
            run("7.2 learning curve", {f"{m} {int(round(f * 100))}%": c})
        for s in SEEDS:
            r = RESULT_BY_KEY[(ckey(c), s)]
            p = PREDS[(ckey(c), s)]["val"][IN_CS["val"]]
            curve.append({"model": m, "fraction": f, "seed": s, "n_train": r["n_train"], "epoch_time_s": r["epoch_time_s"],
                          "val_logloss": metrics(Y_VAL_CS, p)["logloss"]})
curve = pd.DataFrame(curve)
curve_sum = (curve.groupby(["model", "fraction"])
             .agg(n_train=("n_train", "mean"), epoch_time_s=("epoch_time_s", "mean"),
                  val_logloss=("val_logloss", "mean"), logloss_sd=("val_logloss", "std")).reset_index())
display(curve_sum.round(4))
curve.to_csv(RESULTS_DIR / "scaling_learning_curve.csv", index=False)

def power_law(n, a, b, c):
    return a + b * (n / 1e4) ** (-c)

FIT = {}
for m in FINAL:
    d = curve_sum[curve_sum["model"] == m].sort_values("n_train")
    n, y = d["n_train"].values.astype(float), d["val_logloss"].values
    try:
        FIT[m], _ = curve_fit(power_law, n, y, p0=[max(y.min() - 0.005, 0.0), 0.01, 0.5],
                              bounds=([0, 0, 0.05], [y.min(), 1.0, 3.0]), maxfev=20000)
    except Exception as e:
        FIT[m] = None; print(f"power-law fit failed for the {m} model: {e}")

N_NOW = {m: float(curve_sum[(curve_sum["model"] == m) & (curve_sum["fraction"] == 1.0)]["n_train"].iloc[0]) for m in FINAL}
# uncertainty of the fit: resample the seeds at every training size and refit (200 times)
rng_lc, LC_BAND = np.random.default_rng(0), {}
for m in FINAL:
    if FIT[m] is None:
        continue
    d = curve[curve["model"] == m]
    fr = sorted(d["fraction"].unique())
    ns = np.array([d[d["fraction"] == f]["n_train"].mean() for f in fr], dtype=float)
    per = [d[d["fraction"] == f]["val_logloss"].values for f in fr]
    draws = {k: [] for k in (1, 10, 100, 1000)}; asym = []
    for _ in range(200):
        y = np.array([rng_lc.choice(v, len(v)).mean() for v in per])
        try:
            p, _ = curve_fit(power_law, ns, y, p0=[max(y.min() - 0.005, 0.0), 0.01, 0.5],
                             bounds=([0, 0, 0.05], [y.min(), 1.0, 3.0]), maxfev=20000)
        except Exception:
            continue
        asym.append(p[0])
        for k in draws:
            draws[k].append(power_law(N_NOW[m] * k, *p))
    LC_BAND[m] = {k: np.percentile(v, [5, 95]) for k, v in draws.items() if len(v) > 10}
    LC_BAND[m]["asym"] = np.percentile(asym, [5, 95]) if len(asym) > 10 else np.array([np.nan, np.nan])

rows = []
for m in FINAL:
    if FIT[m] is None:
        continue
    for k in [1, 10, 100, 1000]:
        band = LC_BAND[m].get(k, np.array([np.nan, np.nan]))
        rows.append({"model": m, "data volume": f"×{k}", "predicted val log loss": power_law(N_NOW[m] * k, *FIT[m]),
                     "5th pct (refits)": band[0], "95th pct (refits)": band[1], "asymptote a": FIT[m][0]})
PRED_LC = pd.DataFrame(rows)
display(PRED_LC.round(4))

gap = (curve_sum.pivot(index="fraction", columns="model", values=["val_logloss", "logloss_sd", "n_train"]))
gap_df = pd.DataFrame({"training shots": gap[("n_train", "set")], "graph − set": gap[("val_logloss", "graph")] - gap[("val_logloss", "set")],
                       "sd (seeds)": np.sqrt(gap[("logloss_sd", "graph")] ** 2 + gap[("logloss_sd", "set")] ** 2)})

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for m, c in [("set", "tab:orange"), ("graph", "tab:blue")]:
    d = curve_sum[curve_sum["model"] == m].sort_values("n_train")
    ax[0].plot(d["n_train"], d["epoch_time_s"], "o-", color=c, label=m)
    ax[1].errorbar(d["n_train"], d["val_logloss"], yerr=d["logloss_sd"].fillna(0), fmt="o", color=c, capsize=3, label=m)
    if FIT[m] is not None:
        xs = np.logspace(np.log10(d["n_train"].min()), np.log10(N_NOW[m] * 1000), 80)
        ax[1].plot(xs, power_law(xs, *FIT[m]), "--", color=c, alpha=0.7, label=f"{m}: power-law fit")
ax[0].set(xlabel="training shots", ylabel="seconds per epoch", title="Time per epoch vs data size"); ax[0].legend()
ax[1].axhline(SB_VAL, color="k", ls="--", lw=1, label="StatsBomb xG"); ax[1].set_xscale("log")
ax[1].set(xlabel="training shots (log scale, fit extended to ×1000)", ylabel="validation log loss", title="Accuracy vs data size")
ax[1].legend(fontsize=7)
ax[2].errorbar(gap_df["training shots"], gap_df["graph − set"], yerr=gap_df["sd (seeds)"].fillna(0), fmt="o-", color="tab:purple", capsize=3)
ax[2].axhline(0, color="k", lw=1); ax[2].set_xscale("log")
ax[2].set(xlabel="training shots (log scale)", ylabel="log loss(graph) − log loss(set)", title="Does the graph model catch up with more data?")
fig.suptitle("Fig 13. Learning curve (mean ± sd over seeds)", fontsize=10)
plt.tight_layout(); plt.savefig(FIGURES / "fig13_learning_curve.png", dpi=150); plt.show()

### 7.3 Time against data volume
One training epoch over k copies of the training set (×1, ×2, ×5, ×10). Batches are built on the fly from the stored graphs, so memory
does not grow with k: only the index list does. The cost of a batch depends only on the number of nodes and edges, not on whether the
shots are new, so copies give an honest *timing*. We also report how much memory the graphs would need if all of them were held in
memory at once. We stop at ×10 because the trend is already clearly linear (see the fit quality below); going to ×1000 would
only measure the same line for hours.

In [ ]:
def time_epoch(cfg, idx, max_batches=None):
    torch.manual_seed(0)
    store = get_store(cfg["rule"], cfg["r_front"], cfg["r_back"])
    model = make_model(cfg).to(DEVICE); model.train()
    opt, lossf = torch.optim.AdamW(model.parameters(), lr=1e-3), nn.BCEWithLogitsLoss()
    sync(); t0, nb = time.time(), 0
    for i in range(0, len(idx), 256):
        b = store.batch(idx[i:i + 256], cfg["feats"]).to(DEVICE)
        opt.zero_grad(); lossf(model(b), b.y).backward(); opt.step(); nb += 1
        if max_batches and nb >= max_batches:
            break
    sync()
    return time.time() - t0

def bytes_per_shot(cfg):
    store = get_store(cfg["rule"], cfg["r_front"], cfg["r_back"])
    nodes = NCOUNT.mean() * (X_FEATS[cfg["feats"]].shape[1] + 2) * 4     # features + x, y (float32)
    edges = store.mean_edges * (2 * 2 + 4 * 4)                            # src, dst (int16) + 4 edge features
    return nodes + edges + U.shape[1] * 4

synth = []
for k in S_["synth"]:
    for m, cfg in FINAL.items():
        idx = np.random.default_rng(k).permutation(np.tile(train_indices(cfg), k))
        t = time_epoch(cfg, idx)
        synth.append({"model": m, "copies": k, "shots": len(idx), "epoch_time_s": t,
                      "memory_if_all_in_RAM_GB": len(idx) * bytes_per_shot(cfg) / 1e9})
        print(f"  ×{k:<3} {m:<6} {len(idx):>8,} shots: {t:6.1f} s per epoch")
synth = pd.DataFrame(synth)

FITS_T, R2_T = {}, {}
for m, d in synth.groupby("model"):
    FITS_T[m] = np.polyfit(d["shots"], d["epoch_time_s"], 1)                      # time = a * shots + b
    res = d["epoch_time_s"] - np.polyval(FITS_T[m], d["shots"])
    R2_T[m] = 1 - (res ** 2).sum() / max(((d["epoch_time_s"] - d["epoch_time_s"].mean()) ** 2).sum(), 1e-12)
display(synth.round(3))
print("straight-line fit quality (R²):", {m: round(v, 4) for m, v in R2_T.items()})
synth.to_csv(RESULTS_DIR / "scaling_synthetic.csv", index=False)

fig, ax = plt.subplots(figsize=(6.5, 4.2))
for m, c in [("set", "tab:orange"), ("graph", "tab:blue")]:
    d = synth[synth["model"] == m]
    ax.plot(d["shots"], d["epoch_time_s"], "o", color=c, label=f"{m} (measured)")
    xs = np.linspace(0, d["shots"].max(), 50)
    ax.plot(xs, np.polyval(FITS_T[m], xs), "-", color=c, alpha=0.6, label=f"{m}: linear fit (R² {R2_T[m]:.3f})")
ax.set(xlabel="shots per epoch", ylabel="seconds per epoch", title="Fig 14. Time per epoch on ×1 ... ×10 copies of the data")
ax.legend(fontsize=8); plt.tight_layout(); plt.savefig(FIGURES / "fig14_synthetic_scaling.png", dpi=150); plt.show()

### 7.4 Cost per edge rule
From Experiment 5.1: time per epoch against the number of edges per shot. Message passing costs O(edges × hidden²) per round, so the
rule chosen decides how expensive the model is at scale. A straight-line fit checks that claim.

In [ ]:
e1 = exp_table("5.1 edge rule")
slope, icpt = np.polyfit(e1["edges_per_shot"], e1["epoch_time_s"], 1)
res = e1["epoch_time_s"] - (slope * e1["edges_per_shot"] + icpt)
COST_FIT = dict(slope=slope, intercept=icpt, r2=1 - (res ** 2).sum() / ((e1["epoch_time_s"] - e1["epoch_time_s"].mean()) ** 2).sum())
print(f"epoch time ≈ {icpt:.2f} s + {slope:.3f} s per edge-per-shot   (R² {COST_FIT['r2']:.3f})")

fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.scatter(e1["edges_per_shot"], e1["epoch_time_s"], c=e1["model"].map({"set": "tab:orange", "graph": "tab:blue"}), s=60)
xs = np.linspace(0, e1["edges_per_shot"].max(), 50); ax.plot(xs, slope * xs + icpt, "k:", lw=1)
for _, r in e1.iterrows():
    ax.annotate(r["name"], (r["edges_per_shot"], r["epoch_time_s"]), xytext=(5, 3), textcoords="offset points", fontsize=8)
ax.set(xlabel="directed edges per shot", ylabel="seconds per epoch", title="Fig 15. Cost of each edge rule (Experiment 5.1)")
plt.tight_layout(); plt.savefig(FIGURES / "fig15_cost_per_edge_rule.png", dpi=150); plt.show()

### 7.5 What parallelises?
Shots are independent graphs, so training can be split across workers. Two things are tested.

* **(a) Independent worker processes (the compute part of data parallelism).** `W` processes with one thread each, each running complete
  training steps (batch building, forward, backward, optimiser) on its own `1/W` of a fixed amount of work. No gradients are exchanged here;
  that cost is modelled in 7.6. This needs the `fork` start method (Linux/macOS), otherwise it is skipped. Once `W` exceeds the number of
  CPU cores no further speed-up is possible on this machine.
* **(b) Gradient equivalence.** Data parallelism averages the workers' gradients (all-reduce). That equals the single-process gradient *exactly*
  only if shots do not influence each other inside a batch. Our model has no BatchNorm and no edges between shots, so it should. We test this numerically
  (dropout off) with 2 and 4 simulated shards.

In [ ]:
# (a) independent worker processes, each running full training steps on its own shard (no gradient exchange)
def _dp_worker(args):
    cfg, chunks = args
    torch.set_num_threads(1)
    store = get_store(cfg["rule"], cfg["r_front"], cfg["r_back"])         # already built in the parent, inherited by fork
    model = make_model(cfg); model.train()
    opt, lossf = torch.optim.AdamW(model.parameters(), lr=1e-3), nn.BCEWithLogitsLoss()
    for ids in chunks:
        b = store.batch(ids, cfg["feats"]); opt.zero_grad(); lossf(model(b), b.y).backward(); opt.step()
    return len(chunks)

def time_data_parallel(cfg, W, total_batches):
    import multiprocessing as mp
    idx = train_indices(cfg)
    chunks = [idx[i:i + 256] for i in range(0, len(idx), 256)]
    chunks = (chunks * (total_batches // len(chunks) + 1))[:total_batches]            # fixed total amount of work
    shards = [(cfg, chunks[w::W]) for w in range(W)]
    with mp.get_context("fork").Pool(W) as pool:
        pool.map(_dp_worker, [(cfg, chunks[:1])] * W)                                  # warm-up
        t0 = time.perf_counter(); pool.map(_dp_worker, shards)
        return time.perf_counter() - t0

proc_counts = sorted({1, 2} | {w for w in (4, 8) if w <= N_CPU})
dpar = []
try:
    for m, cfg in FINAL.items():
        for W in proc_counts:
            dpar.append({"model": m, "processes": W, "seconds": time_data_parallel(cfg, W, 8 * S_["profile_batches"])})
    dpar = pd.DataFrame(dpar)
    dpar["speed-up"] = dpar.groupby("model")["seconds"].transform(lambda s: s.iloc[0] / s)
    dpar["efficiency"] = dpar["speed-up"] / dpar["processes"]
    dpar["cores available"] = N_CPU
    display(dpar.round(2))
    dpar.to_csv(RESULTS_DIR / "scaling_processes.csv", index=False)
except Exception as e:
    dpar = pd.DataFrame()
    print(f"Process-level test skipped on this machine: {type(e).__name__}: {e}")

# (b) data parallelism: averaged shard gradients == full-batch gradient
def check_data_parallel(cfg, shards=(2, 4), bs=256):
    store = get_store(cfg["rule"], cfg["r_front"], cfg["r_back"])
    idx = train_indices(cfg)[:bs]
    torch.manual_seed(0)
    model = make_model(cfg).eval()                         # eval: dropout off, so the gradient is deterministic
    lossf = nn.BCEWithLogitsLoss()
    def grads(ids):
        model.zero_grad()
        b = store.batch(ids, cfg["feats"]); lossf(model(b), b.y).backward()
        return torch.cat([p.grad.flatten() for p in model.parameters() if p.grad is not None])
    g_full = grads(idx)
    out = {}
    for w in shards:
        parts = np.array_split(idx, w)
        assert len({len(p) for p in parts}) == 1, "shards must be equally sized"
        g_avg = torch.stack([grads(p) for p in parts]).mean(0)
        out[w] = ((g_full - g_avg).abs().max() / g_full.abs().max()).item()
    return out

DP_ERR = {m: check_data_parallel(cfg) for m, cfg in FINAL.items()}
for m, e in DP_ERR.items():
    for w, v in e.items():
        print(f"{m:<6} {w} shards: max relative difference between averaged shard gradients and full-batch gradient = {v:.1e}")
        assert v < 1e-4, "data-parallel gradient does not match the single-process gradient"
print("OK: gradient averaging over shards reproduces the single-process gradient (up to floating-point error).")

if DEVICE.type == "cuda":
    print("GPU available: the single-process timings in this section were run on", torch.cuda.get_device_name(0))

if len(dpar):
    fig, ax = plt.subplots(figsize=(6, 4))
    for m, c in [("set", "tab:orange"), ("graph", "tab:blue")]:
        d = dpar[dpar["model"] == m]; ax.plot(d["processes"], d["speed-up"], "o-", color=c, label=m)
    ax.plot(dpar["processes"].unique(), dpar["processes"].unique(), ":", color="grey", label="perfect scaling")
    ax.axvline(N_CPU, color="red", lw=0.8, ls="--"); ax.text(N_CPU, ax.get_ylim()[0], " cores", color="red", fontsize=7)
    ax.set(xlabel="worker processes (1 thread each)", ylabel="speed-up vs 1 process",
           title="Fig 16. Speed-up from independent worker processes")
    ax.legend(); plt.tight_layout(); plt.savefig(FIGURES / "fig16_parallelism.png", dpi=150); plt.show()


### 7.6 Extrapolating to 1000× the data
**Single machine** (from the straight-line fits of 7.3): time per epoch, the number of epochs the final models needed (best epoch + patience,
which is an upper bound for much larger data), the one-off cost of building every shot's edges (measured while building the graphs,
one core), and the memory needed if everything were held in RAM (it should instead be streamed from disk in shards).

**Cluster model.** With `W` workers each processing a batch of 256 shots, one step takes
`t_step + t_comm(W)`, where `t_step` is the measured single-process step of 7.1 (a laptop-class worker) and
`t_comm` is a ring all-reduce of the gradients: `2(W−1)/W · bytes/bandwidth + 2(W−1)·latency`. The gradient is tiny (a few hundred kilobytes),
so communication is small *relative to a CPU step*. If each worker is a **GPU**, `t_step` shrinks and the same
communication becomes relatively larger, so the table also gives the **break-even step time**: the shortest step for which efficiency
stays at or above 80% (`t_step ≥ 4·t_comm`, no overlap of compute and communication).
The interconnect numbers below are **assumptions** of typical orders of magnitude, not measurements; edit them to match a real cluster.
Not modelled: stragglers, input-pipeline limits, and the fact that a global batch of 256·W shots needs a retuned learning rate and may need
different numbers of epochs.

In [ ]:
extrap_rows = []
EPOCHS = {}
for m, cfg in FINAL.items():
    EPOCHS[m] = min(MAX_EPOCHS, np.mean([RESULT_BY_KEY[(ckey(cfg), s)]["best_epoch"] + PATIENCE for s in SEEDS]))
    key = (cfg["rule"], cfg["r_front"], cfg["r_back"]) if cfg["rule"] in REGION_RULES else (cfg["rule"],)
    build_per_shot_s = BUILD_TIME[key] / S
    n_now = len(train_indices(cfg))
    for k in [1, 10, 100, 1000]:
        sec = np.polyval(FITS_T[m], n_now * k)
        extrap_rows.append({"model": m, "data volume": f"×{k}", "training shots": n_now * k,
                            "hours per epoch (1 machine)": sec / 3600, "epochs": EPOCHS[m],
                            "hours to train (1 machine)": sec * EPOCHS[m] / 3600,
                            "edge building, hours (1 core, one-off)": build_per_shot_s * S * k / 3600,
                            "memory if all in RAM (GB)": n_now * k * bytes_per_shot(cfg) / 1e9})
extrap = pd.DataFrame(extrap_rows)
display(extrap.round(3))
extrap.to_csv(RESULTS_DIR / "scaling_extrapolation.csv", index=False)

# ---- cluster model: ASSUMPTIONS, edit to match a real cluster -------------------------------------
NETWORKS = {"10 GbE cluster": dict(bandwidth_Bps=1.0e9, latency_s=100e-6),
            "NVLink / InfiniBand class": dict(bandwidth_Bps=2.5e10, latency_s=10e-6)}
WORKERS = [1, 8, 64]
def allreduce_s(n_par, W, net):
    return 0.0 if W == 1 else 2 * (W - 1) / W * (4 * n_par) / net["bandwidth_Bps"] + 2 * (W - 1) * net["latency_s"]

rows = []
for m, cfg in FINAL.items():
    n_par = RESULT_BY_KEY[(ckey(cfg), SEEDS[0])]["n_params"]
    N = len(train_indices(cfg)) * 1000
    for net_name, net in NETWORKS.items():
        t1 = None
        for W in WORKERS:
            t_comm = allreduce_s(n_par, W, net)
            hours = N / (256 * W) * (STEP_S[m] + t_comm) / 3600
            t1 = hours if W == 1 else t1
            rows.append({"model": m, "interconnect": net_name, "workers": W, "gradient size (kB)": 4 * n_par / 1e3,
                         "t_step measured (ms)": 1000 * STEP_S[m], "t_comm (ms)": 1000 * t_comm,
                         "efficiency": STEP_S[m] / (STEP_S[m] + t_comm), "hours per epoch at ×1000": hours,
                         "speed-up vs 1 worker": t1 / hours, "break-even t_step for 80% (ms)": 4000 * t_comm if W > 1 else np.nan})
cluster = pd.DataFrame(rows)
display(cluster.round(3))
cluster.to_csv(RESULTS_DIR / "scaling_cluster_model.csv", index=False)

## 8. Conclusions

The text below is **generated from the results above** (this notebook run), so the numbers always match the tables and figures.
Statements about noise follow the rules of Section 4: a difference is only reported as real if its 95% match-bootstrap interval excludes 0.

In [ ]:
fmt_ci = lambda lo, hi: f"[{lo:+.4f}, {hi:+.4f}]"
head = final.loc["test, complete seasons (headline)"]
b_ens, b_one = boot_df.loc["graph − set (ensembles)"], boot_df.loc["graph − set (single models)"]
sb_g, sb_s = boot_df.loc["graph − StatsBomb xG (single models)"], boot_df.loc["set − StatsBomb xG (single models)"]

print("MAIN RESULT (held-out complete-season test matches)")
print(f"  set  : log loss {head.loc['set', 'logloss']:.4f} (single seeds {head.loc['set', 'logloss, single seeds (mean)']:.4f}), AUC {head.loc['set', 'auc']:.3f}")
print(f"  graph: log loss {head.loc['graph', 'logloss']:.4f} (single seeds {head.loc['graph', 'logloss, single seeds (mean)']:.4f}), AUC {head.loc['graph', 'auc']:.3f}")
print(f"  graph − set, ensembles      : {b_ens['log loss difference']:+.4f} {fmt_ci(b_ens['95% CI low'], b_ens['95% CI high'])} -> {b_ens['verdict']}")
print(f"  graph − set, single models  : {b_one['log loss difference']:+.4f} {fmt_ci(b_one['95% CI low'], b_one['95% CI high'])} -> {b_one['verdict']}")
print(f"  vs StatsBomb xG ({head.loc['StatsBomb xG', 'logloss']:.4f}), single models: graph {sb_g['log loss difference']:+.4f} {fmt_ci(sb_g['95% CI low'], sb_g['95% CI high'])} -> {sb_g['verdict']};"
      f" set {sb_s['log loss difference']:+.4f} {fmt_ci(sb_s['95% CI low'], sb_s['95% CI high'])} -> {sb_s['verdict']}")

print("\nWHAT THE EXPERIMENTS SHOWED (validation; every configuration trained with", len(SEEDS), "seeds)")
for (exp, model), c in CHOICES.items():
    print(f"  {exp} ({model}): best by mean = {c['best']}; {len(c['tied'])} of {c['n']} configurations are not distinguishable from it; kept: {c['chosen']}")
better = VS_SET[VS_SET["verdict"] != "no clear difference"]
print("  5.1: graph rules distinguishable from the set model: " +
      (", ".join(f"{i} ({r['verdict']}, {r['graph − set']:+.4f})" for i, r in better.iterrows()) if len(better) else "none"))
for m, (d, lo, hi) in FEAT_GAIN.items():
    print(f"  5.4: engineered features lower val log loss by {d:+.4f} {fmt_ci(lo, hi)} for the {m} model (positive = better) "
          f"({'distinguishable' if (lo > 0 or hi < 0) else 'not distinguishable'} from 0)")
for _, r in TRAIN_CMP.iterrows():
    print(f"  5.5: {r['model']}, {r['comparison']} ({r['question']}): {r['diff']:+.4f} {fmt_ci(r['ci_low'], r['ci_high'])} "
          f"-> {'distinguishable' if r['distinguishable'] else 'not distinguishable'}")
fc = over.loc["single-club: focus club"]
if fc["shots"] > 0:
    print(f"  Fig 9: the focus clubs scored {fc['StatsBomb']:+.1f} goals more than StatsBomb xG expected "
          f"(set {fc['set, trained on all seasons']:+.1f}, graph {fc['graph, trained on all seasons']:+.1f}) over {int(fc['shots'])} validation shots.")

print("\nSCALING")
ratio = (FITS_T['graph'][0] / FITS_T['set'][0]) if FITS_T['set'][0] > 0 else float('nan')
print(f"  Time per epoch is linear in data volume (R² set {R2_T['set']:.3f}, graph {R2_T['graph']:.3f}); the graph model costs {ratio:.1f}× the set model per shot.")
print(f"  Edge cost: epoch time ≈ {COST_FIT['intercept']:.2f} s + {COST_FIT['slope']:.3f} s per edge-per-shot (R² {COST_FIT['r2']:.3f}).")
print("  Time of one step: " + "; ".join(f"{m}: " + ", ".join(f"{k.replace('_', ' ')} {v:.0f}%" for k, v in PROFILE_PCT.loc[m].items() if v >= 5) for m in FINAL) + ".")
e1000 = extrap[extrap["data volume"] == "×1000"].set_index("model")
for m in FINAL:
    print(f"  ×1000 ({m}): {e1000.loc[m, 'hours to train (1 machine)']:.1f} h to train on one machine "
          f"({e1000.loc[m, 'hours per epoch (1 machine)']:.2f} h per epoch × {e1000.loc[m, 'epochs']:.0f} epochs), "
          f"{e1000.loc[m, 'edge building, hours (1 core, one-off)']:.1f} h one-off edge building, "
          f"{e1000.loc[m, 'memory if all in RAM (GB)']:.0f} GB if held in RAM, so stream it from disk.")
if len(dpar) and dpar["processes"].max() > 1:
    q = dpar[dpar["processes"] == dpar["processes"].max()].set_index("model")
    print(f"  Independent worker processes ({N_CPU} CPU cores here): with {int(dpar['processes'].max())} processes the compute speed-up was "
          f"{q.loc['set', 'speed-up']:.2f}× (set) and {q.loc['graph', 'speed-up']:.2f}× (graph)"
          + (" (limited by the number of cores)." if dpar["processes"].max() > N_CPU else "."))
print("  Data parallelism: averaged shard gradients equal the single-process gradient to within floating-point error "
      f"(worst relative difference {max(v for e in DP_ERR.values() for v in e.values()):.1e}).")
cm = cluster[(cluster["workers"] == max(WORKERS))]
for _, r in cm.iterrows():
    print(f"  Cluster model, {r['model']}, {int(r['workers'])} workers, {r['interconnect']}: efficiency {r['efficiency']:.2f}, "
          f"{r['hours per epoch at ×1000']:.2f} h per epoch at ×1000 (assumed network; step time must stay above {r['break-even t_step for 80% (ms)']:.1f} ms for ≥80% efficiency).")
if PRED_LC is not None and len(PRED_LC):
    for m in FINAL:
        d = PRED_LC[PRED_LC["model"] == m].set_index("data volume")
        print(f"  Learning curve ({m}): fitted {d.loc['×1', 'predicted val log loss']:.4f} now, {d.loc['×1000', 'predicted val log loss']:.4f} at ×1000 "
              f"(refit range {d.loc['×1000', '5th pct (refits)']:.4f} to {d.loc['×1000', '95th pct (refits)']:.4f}; fitted asymptote {d.iloc[0]['asymptote a']:.4f}). "
              f"Indicative only: a handful of points cannot pin a power law down.")
print("  Set–graph gap against data size (graph − set): " + ", ".join(f"{int(r['training shots'])} shots: {r['graph − set']:+.4f}±{r['sd (seeds)']:.4f}" for _, r in gap_df.iterrows()))

### Limitations and what we would do next
* **Search.** Tuning was greedy (one factor at a time), so interactions between settings can be missed. Selection uses a validation set that is also used for
  early stopping, which is why the final claims are made on a separate test set that was evaluated once.
* **Noise.** Three seeds per configuration is enough to see large effects but gives only a rough estimate of the seed spread; the match-bootstrap interval
  covers the sample of matches, not a different season or league.
* **Data.** One open-data source; freeze frames show only the *visible* players; no tracking data (player speeds, body orientation, previous movement).
  The shooter's role is not recorded in the frame.
* **Benchmark.** StatsBomb xG is a published, differently trained model, so its gap to ours is not a like-for-like comparison.
* **Scaling.** Everything marked "measured" was measured on the machine that ran this notebook (CPU unless a GPU is reported); ×1000 numbers are extrapolations
  from ×1–×10 timings, and the cluster table is a model with stated assumptions, not a cluster run. Large-batch effects (learning-rate retuning, number of epochs) are not measured.
* **Next.** The most informative extra experiments would be a model-capacity sweep at larger data (the set–graph gap in Fig 13 shows whether the graph model starts
  to pay off), a run on a GPU, and a dataset where relations between players carry more information than in a single frozen frame, such as tracking data.